导入必要的库

In [7]:
import re
from collections import defaultdict, Counter

创建Tokenizer类，主要分为三个函数，train、encode、decode，还有一些其他的功能函数，并配备有详细注释

In [ ]:
import re
from collections import defaultdict, Counter

class Tokenizer:
    def __init__(self):
        # 存储最终词汇表（词元序列 -> 频率）
        self.vocab = {}

        # 存储合并顺序，用于编码时应用相同的BPE合并
        self.merges = []

        # 子词到整数ID的映射
        self.token2id = {}

        # ID到子词的映射
        self.id2token = {}

    def get_stats(self, vocab):
        """
        统计所有词中，相邻 token（二元组）对的出现频率
        """
        pairs = defaultdict(int)
        for word, freq in vocab.items():
            symbols = word  # 每个词是一个由符号组成的元组
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i+1])] += freq  # 统计相邻对的频率（乘以该词频）
        return pairs

    def merge_vocab(self, pair, vocab):
        """
        对vocab中的所有词，合并指定的 pair（二元组），返回新的词表
        """
        new_vocab = {}

        # 将 pair 转换为字符串形式，用于正则匹配
        bigram = re.escape(' '.join(pair))

        # 匹配单词中的 pair，但要求其是独立的token（避免子串误匹配）
        pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')

        for word, freq in vocab.items():
            word_str = ' '.join(word)  # 把元组还原为字符串
            # 替换掉目标pair为合并后的token
            new_word = tuple(pattern.sub(''.join(pair), word_str).split())
            new_vocab[new_word] = freq  # 更新词表
        return new_vocab

    def train(self, text, vocab_size):
        """
        使用BPE算法训练tokenizer
        :param text: 输入语料
        :param vocab_size: 最终词汇表大小
        """
        # 文本拆词（按空格），然后将每个词转换为字+结束符</w>，以区分词边界
        words = text.strip().split()
        vocab = Counter([' '.join(list(word)) + ' </w>' for word in words])

        # 将词表示为符号元组，构建初始vocab
        self.vocab = {tuple(word.split()): freq for word, freq in vocab.items()}

        # BPE迭代合并，直到达到词汇表大小上限
        while len(self.vocab) + len(self.merges) < vocab_size:
            pairs = self.get_stats(self.vocab)
            if not pairs:
                break
            # 选出频率最高的符号对
            best = max(pairs, key=pairs.get)
            self.merges.append(best)  # 记录合并历史
            # 在词表中合并这个pair
            self.vocab = self.merge_vocab(best, self.vocab)

        # 构建token -> id 映射
        tokens = set()
        for word in self.vocab:
            tokens.update(word)
        self.token2id = {token: idx for idx, token in enumerate(sorted(tokens))}
        self.id2token = {idx: token for token, idx in self.token2id.items()}

    def encode_word(self, word):
        """
        对单个词应用 BPE 合并规则，编码为 token id 序列
        """
        # 初始符号序列：每个字母 + </w> 结尾
        symbols = list(word) + ['</w>']

        # 构造快速查找的合并对排名
        merges = {pair: i for i, pair in enumerate(self.merges)}

        while True:
            # 生成当前符号列表中所有相邻的二元组
            pairs = [(symbols[i], symbols[i+1]) for i in range(len(symbols)-1)]
            # 保留在 merges 中存在的对，并带上合并顺序编号
            pair_ranks = [(p, merges[p]) for p in pairs if p in merges]
            if not pair_ranks:
                break
            # 选择最优（优先级最高）的合并对
            best_pair = min(pair_ranks, key=lambda x: x[1])[0]

            # 执行合并操作
            new_symbols = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i+1]) == best_pair:
                    # 合并这个对
                    new_symbols.append(symbols[i] + symbols[i+1])
                    i += 2  # 跳过下一个
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols  # 更新符号序列

        # 将符号转换为id，过滤未出现在词表中的（极少见）
        return [self.token2id[sym] for sym in symbols if sym in self.token2id]

    def encode(self, text):
        """
        对输入文本进行编码，输出token id序列
        """
        tokens = []
        for word in text.strip().split():
            tokens.extend(self.encode_word(word))  # 编码每个词
        return tokens

    def decode(self, ids):
        """
        将token id序列解码为原始字符串
        """
        tokens = [self.id2token[i] for i in ids]  # 查找token字符串
        text = ' '.join(tokens)
        text = text.replace('</w>', '')  # 去除词尾标记
        return text.replace(' ', '')  # 移除分词空格


创建实例，给出示例文本，训练、编码、解码

In [15]:
tokenizer = Tokenizer()
tokenizer.train("low lower lowest", vocab_size=50)
print("Encoded:", tokenizer.encode("lowest"))
print("Decoded:", tokenizer.decode(tokenizer.encode("lowest")))

Encoded: [2]
Decoded: lowest


In [16]:
# 读取manual.txt文件
with open("ref/manual.txt", "r", encoding="utf-8") as f:
    manual_text = f.read()

# 输出读取的文本内容
# print(manual_text)

# 设置词汇表的大小，训练tokenizer
print("Training tokenizer...")
tokenizer.train(manual_text, vocab_size=1024)
print("Tokenizer trained successfully.")
# print("Vocabulary:", tokenizer.vocab)

# 将字符串编码为token
# encoded = tokenizer.encode("newest")
encoded = tokenizer.encode(manual_text)
print(f"Encoded: {encoded}")

# 解码回原始字符串
decoded = tokenizer.decode(encoded)
print(f"Decoded: {decoded}")

if decoded == manual_text:
    print("Decoded text matches the original manual text.")
else:
    print("Decoded text does NOT match the original manual text.")

Training tokenizer...
Tokenizer trained successfully.
Encoded: [345, 21, 21, 172, 21, 21, 494, 21, 21, 532, 21, 21, 38, 27, 33, 31, 36, 29, 21, 21, 43, 36, 31, 44, 27, 40, 41, 31, 42, 47, 21, 21, 345, 21, 21, 172, 21, 21, 95, 21, 21, 532, 21, 21, 1126, 21, 21, 1163, 21, 21, 102, 21, 21, 99, 21, 21, 282, 21, 21, 1646, 21, 21, 11, 21, 21, 9, 21, 21, 11, 21, 21, 12, 21, 21, 1045, 21, 21, 1647, 21, 21, 345, 21, 21, 172, 21, 21, 95, 21, 21, 532, 21, 21, 1126, 21, 21, 1163, 21, 21, 102, 21, 21, 1574, 21, 21, 11, 21, 21, 9, 21, 21, 11, 21, 21, 12, 21, 21, 612, 21, 21, 17, 21, 21, 101, 21, 21, 152, 1473, 611, 562, 1126, 1163, 1073, 815, 1267, 593, 220, 220, 300, 1528, 1334, 766, 1138, 644, 1391, 21, 21, 1490, 621, 266, 423, 451, 550, 160, 130, 386, 584, 1587, 1334, 21, 21, 473, 279, 1499, 574, 494, 733, 664, 721, 280, 488, 1100, 1631, 580, 924, 175, 721, 21, 21, 1646, 11, 9, 11, 9, 21, 21, 612, 21, 21, 16, 21, 21, 865, 1647, 21, 21, 136, 272, 136, 498, 681, 153, 1356, 108, 451, 550, 140, 606, 

加载transformers中预训练好的gpt2以及tokenizer，可以直接调用；
定义题目要求的两个句子，并使用GPT-2的tokenizer和我自己定义的tokenizer分别进行编码
最后打印出编码后的结果

In [17]:
# 加载huggingface transformers中的tokenizer
from transformers import GPT2Tokenizer

# 加载GPT-2的tokenizer
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# 定义句子
sentence_1 = "Originated as the Imperial University of Peking in 1898, Peking University was China’s first national comprehensive university and the supreme education authority at the time. Since the founding of the People’s Republic of China in 1949, it has developed into a comprehensive university with fundamental education and research in both humanities and science. The reform and opening-up of China in 1978 has ushered in a new era for the University unseen in history. And its merger with Beijing Medical University in 2000 has geared itself up for all-round and vibrant growth in such fields as science, engineering, medicine, agriculture, humanities and social sciences. Supported by the “211 Project” and the “985 Project”, the University has made remarkable achievements, such as optimizing disciplines, cultivating talents, recruiting high-caliber teachers, as well as teaching and scientific research, which paves the way for a world-class university."
sentence_2 = "博士学位论文应当表明作者具有独立从事科学研究工作的能力，并在科学或专门技术上做出创造性的成果。博士学位论文或摘要，应当在答辩前三个月印送有关单位，并经同行评议。学位授予单位应当聘请两位与论文有关学科的专家评阅论文，其中一位应当是外单位的专家。评阅人应当对论文写详细的学术评语，供论文答辩委员会参考。"

# 使用GPT-2的tokenizer进行编码
gpt2_encoded_1 = gpt2_tokenizer.encode(sentence_1)
gpt2_encoded_2 = gpt2_tokenizer.encode(sentence_2)

# 使用自己实现的tokenizer进行编码
custom_encoded_1 = tokenizer.encode(sentence_1)
custom_encoded_2 = tokenizer.encode(sentence_2)
custom_decoded_1 = tokenizer.decode(custom_encoded_1)
custom_decoded_2 = tokenizer.decode(custom_encoded_2)

# 输出编码后的结果
print("############### GPT-2 Tokenizer Results ###############")
print("############### Sentence 1 ###############")
print(f"GPT-2 Tokenizer - Sentence 1: Length = {len(gpt2_encoded_1)}")
print(f"GPT-2 Tokenizer - Sentence 1: Tokens = {gpt2_tokenizer.decode(gpt2_encoded_1)}")
print(f"GPT-2 Tokenizer - Sentence 1 (Token IDs): {gpt2_encoded_1}")
print(f"Custom Tokenizer - Sentence 1: Length = {len(custom_encoded_1)}")
print(f"Custom Tokenizer - Sentence 1: Tokens = {custom_decoded_1}")
print(f"Custom Tokenizer - Sentence 1 (Token IDs): {custom_encoded_1}")

print("############### Sentence 2 ###############")
print(f"GPT-2 Tokenizer - Sentence 2: Length = {len(gpt2_encoded_2)}")
print(f"GPT-2 Tokenizer - Sentence 2: Tokens = {gpt2_tokenizer.decode(gpt2_encoded_2)}")
print(f"GPT-2 Tokenizer - Sentence 2 (Token IDs): {gpt2_encoded_2}")
print(f"Custom Tokenizer - Sentence 2: Length = {len(custom_encoded_2)}")
print(f"Custom Tokenizer - Sentence 2: Tokens = {custom_decoded_2}")
print(f"Custom Tokenizer - Sentence 2 (Token IDs): {custom_encoded_2}")

d:\Research\anaconda3\envs\llmba\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


############### GPT-2 Tokenizer Results ###############
############### Sentence 1 ###############
GPT-2 Tokenizer - Sentence 1: Length = 185
GPT-2 Tokenizer - Sentence 1: Tokens = Originated as the Imperial University of Peking in 1898, Peking University was China’s first national comprehensive university and the supreme education authority at the time. Since the founding of the People’s Republic of China in 1949, it has developed into a comprehensive university with fundamental education and research in both humanities and science. The reform and opening-up of China in 1978 has ushered in a new era for the University unseen in history. And its merger with Beijing Medical University in 2000 has geared itself up for all-round and vibrant growth in such fields as science, engineering, medicine, agriculture, humanities and social sciences. Supported by the “211 Project” and the “985 Project”, the University has made remarkable achievements, such as optimizing disciplines, cultivating tal